# SpaceX Falcon 9 — Data Collection with the SpaceX REST API
Collect past launches from `api.spacexdata.com/v4`, enrich with helper calls, keep Falcon 9 only and impute missing payload mass.

In [ ]:
import requests, datetime
import pandas as pd, numpy as np
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [ ]:
# helper functions: one extra API call per ID
def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(x)).json()
            BoosterVersion.append(response['name'])

def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(x)).json()
            Longitude.append(response['longitude']); Latitude.append(response['latitude']); LaunchSite.append(response['name'])

def getPayloadData(data):
    for load in data['payloads']:
        if load:
            response = requests.get("https://api.spacexdata.com/v4/payloads/"+load).json()
            PayloadMass.append(response['mass_kg']); Orbit.append(response['orbit'])

def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            response = requests.get("https://api.spacexdata.com/v4/cores/"+core['core']).json()
            Block.append(response['block']); ReusedCount.append(response['reuse_count']); Serial.append(response['serial'])
        else:
            Block.append(None); ReusedCount.append(None); Serial.append(None)
        Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
        Flights.append(core['flight']); GridFins.append(core['gridfins']); Reused.append(core['reused'])
        Legs.append(core['legs']); LandingPad.append(core['landpad'])

In [ ]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print(response.status_code)
# IBM static copy (identical structure) - use it so results match the course
static_json_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'
response = requests.get(static_json_url)
data = pd.json_normalize(response.json())
data.head()

In [ ]:
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
data = data[data['cores'].map(len)==1]
data = data[data['payloads'].map(len)==1]
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]

In [ ]:
BoosterVersion, PayloadMass, Orbit, LaunchSite, Outcome = [], [], [], [], []
Flights, GridFins, Reused, Legs, LandingPad = [], [], [], [], []
Block, ReusedCount, Serial, Longitude, Latitude = [], [], [], [], []
getBoosterVersion(data); getLaunchSite(data); getPayloadData(data); getCoreData(data)
launch_dict = {'FlightNumber': list(data['flight_number']), 'Date': list(data['date']),
 'BoosterVersion': BoosterVersion, 'PayloadMass': PayloadMass, 'Orbit': Orbit, 'LaunchSite': LaunchSite,
 'Outcome': Outcome, 'Flights': Flights, 'GridFins': GridFins, 'Reused': Reused, 'Legs': Legs,
 'LandingPad': LandingPad, 'Block': Block, 'ReusedCount': ReusedCount, 'Serial': Serial,
 'Longitude': Longitude, 'Latitude': Latitude}
launch_df = pd.DataFrame(launch_dict)
launch_df.head()

In [ ]:
# keep Falcon 9 only and renumber flights
data_falcon9 = launch_df[launch_df['BoosterVersion'] != 'Falcon 1'].copy()
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
# impute missing PayloadMass with the mean
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].fillna(data_falcon9['PayloadMass'].mean())
print(data_falcon9.shape)
data_falcon9.isnull().sum()

**Result:** 90 Falcon 9 launches; PayloadMass has no missing values after imputation, LandingPad stays missing where no pad was used.

In [ ]:
data_falcon9.to_csv('dataset_part_1.csv', index=False)